# Anarkali: scenario testing + training notebook

This notebook is meant for practical evaluation and iteration.
It includes:
- loading the released model
- running realistic decision scenarios
- summarizing confidence and probability behavior
- preparing typed training data
- training a stronger checkpoint in Colab or GPU runtime
- benchmarking the trained model

Use this in VS Code or in a Colab runtime.

In [1]:
# 1) Setup
import json
from pathlib import Path

try:
    from anarkali import Engine
    print("Anarkali import OK")
except Exception as exc:
    print(f"Import error: {exc}")
    raise

root = Path.cwd()
print("Working directory:", root)

Import error: No module named 'anarkali'


ModuleNotFoundError: No module named 'anarkali'

In [ ]:
# 2) Load released model
engine = Engine.load("release-150m")
print("Loaded model:", engine.name)

In [ ]:
# 3) Realistic decision scenarios
scenarios = {
    "support_routing": {
        "state": {
            "customer": "A customer reports they cannot access the portal after password reset.",
            "context": ["MFA required", "user can login to mobile but not desktop"],
            "severity": "medium"
        },
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which route best matches this issue?",
                "criteria": {
                    "identity": "Identity or account access problem",
                    "security": "Security incident or breach",
                    "product": "Product bug or feature defect",
                    "billing": "Billing or payment issue"
                }
            }
        }
    },
    "ci_failure": {
        "state": {
            "repo": "payments-service",
            "error": "pytest fails on invoice export job",
            "logs": ["KeyError: 'currency'", "job started after deploy"]
        },
        "questions": {
            "cause": {
                "type": "choice",
                "instructions": "What is the most likely cause of the CI failure?",
                "criteria": {
                    "schema_change": "Recent schema or contract change",
                    "environment": "Environment or dependency issue",
                    "test_flake": "Test flake or nondeterministic failure",
                    "infra": "Infrastructure or runner issue"
                }
            }
        }
    },
    "invoice_approval": {
        "state": {
            "invoice": "INV-2026-0917",
            "amount": 48120,
            "currency": "INR",
            "approval_policy": "Invoices above 40000 INR require finance approval before payment."
        },
        "questions": {
            "needs_finance_approval": {
                "type": "noul",
                "instructions": "The invoice needs finance approval before payment.",
                "criteria": {
                    "false": "The invoice can be paid without finance approval.",
                    "true": "The invoice must be approved by finance before payment."
                }
            }
        }
    },
    "risk_level": {
        "state": {
            "incident": "API latency spiked for 10 minutes across 2 regions.",
            "evidence": ["customer complaints", "partial outage", "no confirmed breach"]
        },
        "questions": {
            "risk": {
                "type": "score",
                "instructions": "Estimate the operational risk level.",
                "criteria": ["low", "medium", "high", "critical"]
            }
        }
    },
    "product_vs_incident": {
        "state": {
            "description": "A dozen users report missing export button after the latest release.",
            "activity": ["single feature affected", "no signs of broader outage"]
        },
        "questions": {
            "classification": {
                "type": "choice",
                "instructions": "How should this issue be classified?",
                "criteria": {
                    "product": "Product bug or regression",
                    "incident": "Widespread service incident",
                    "security": "Security problem",
                    "billing": "Billing problem"
                }
            }
        }
    },
    "suspicious_login": {
        "state": {
            "customer": "User reports login from a new device in Singapore despite being in Mumbai.",
            "signals": ["same session token used", "MFA passed", "no malware report"],
            "risk": "medium"
        },
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which handling path is best?",
                "criteria": {
                    "identity": "Account identity verification / suspicious login",
                    "security": "Active security incident or breach",
                    "product": "Product defect or feature issue",
                    "billing": "Billing or payment problem"
                }
            }
        }
    },
    "refund_request": {
        "state": {
            "customer": "Customer wants refund after duplicate charge on monthly subscription.",
            "amount": 299,
            "status": "paid",
            "history": ["first charge succeeded", "second charge appears across 2 subscriptions"]
        },
        "questions": {
            "manual_review_needed": {
                "type": "noul",
                "instructions": "The refund request likely requires manual finance review.",
                "criteria": {
                    "false": "The case can be handled automatically.",
                    "true": "The case should be escalated for manual review."
                }
            }
        }
    },
    "platform_outage": {
        "state": {
            "incident": "Checkout API latency increased by 9x for 20 minutes and 30% of checkouts failed.",
            "signals": ["customer impact across 3 regions", "no confirmed breach", "recent deploy happened 40 minutes ago"],
            "owner": "platform"
        },
        "questions": {
            "risk": {
                "type": "score",
                "instructions": "Rate the severity of the operational incident.",
                "criteria": ["low", "medium", "high", "critical"]
            }
        }
    },
    "data_exposure": {
        "state": {
            "description": "Support ticket says a third-party script accessed user export data without consent.",
            "details": ["export logs show 4k records", "customer reported no unauthorized login"],
            "sensitivity": "high"
        },
        "questions": {
            "classification": {
                "type": "choice",
                "instructions": "How should this problem be classified?",
                "criteria": {
                    "security": "Security or privacy incident",
                    "product": "Product bug or feature issue",
                    "identity": "Identity or access issue",
                    "billing": "Billing or payment issue"
                }
            }
        }
    }
}

for name, payload in scenarios.items():
    print(f"\n=== {name} ===")
    result = engine.predict(payload["state"], payload["questions"])
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# 4) Compact summary function

def summarize_answer(answer):
    kind = answer["type"]
    if kind == "choice":
        return {
            "picked": answer["choice"],
            "confidence": answer["confidence"],
            "top_prob": max(answer["probabilities"].values()),
            "type": kind,
        }
    if kind == "noul":
        return {
            "prob_true": answer["noul"],
            "confidence": answer["confidence"],
            "type": kind,
        }
    return {
        "score": answer["score"],
        "confidence": answer["confidence"],
        "type": kind,
    }

for name, payload in scenarios.items():
    result = engine.predict(payload["state"], payload["questions"])
    summary = {qid: summarize_answer(answer) for qid, answer in result["answers"].items()}
    print(f"\n{name}: {summary}")

## Interpretation

- `choice`: picks the strongest option and shows the probability split.
- `noul`: gives the probability that the statement is true.
- `score`: gives a value on an ordered risk/regret scale.
- `confidence`: a rough indication of how certain the model is.

If confidence is low, the safe behavior is human review instead of autonomous action.

In [ ]:
# 5) Prepare typed decision data (repo-native pipeline)
!python scripts/prepare_typed_decisions.py --question-types choice,noul,score --output artifacts/focused-retrain-v1

In [ ]:
# 6) Train a stronger checkpoint
# This is the repo-native training entrypoint.
# For Colab/GPU use `--device cuda` and a bigger workload.
# Example:
# !python scripts/train_anarkali.py --data artifacts/focused-retrain-v1 --output artifacts/colab-heavy-run --epochs 4 --batch-size 16 --evaluate-test --device cuda --max-train-rows 2000 --max-dev-rows 300

print("Training command is ready to run. Use the cell above as the template.")

In [ ]:
# 7) Optional benchmark check on the released model
# !python scripts/evaluate_release_suite.py --model release-150m --cases benchmarks/anarkali-routing-v1/cases.jsonl --min-accuracy 0.75

print("Benchmark command is ready to run.")

## Colab instructions

If you are using Google Colab:

```bash
!git clone https://github.com/ToufiqQureshi/anarkali.git
%cd anarkali
!python scripts/colab_train.py --epochs 4 --batch-size 16 --device cuda --max-train-rows 2000 --max-dev-rows 300
```

This uses the project’s native training workflow and is the best next step for actual improvement.